<a href="https://colab.research.google.com/github/ankitsingh435517/nn/blob/main/makemore5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch # imports torch lib
import torch.nn.functional as F # imports functional sub lib of torch.nn module
import matplotlib.pyplot as plt # imports pyplot lib from matplotlib as plot for drawing figures (used for debugging and visualizing losses and other training related things)
 # it's a magic command for jupyter and collab notebooks which tells matplotlib plots to render below the executing cells instead of in a seperate window (matplotlib used to open new windows in its interface)
%matplotlib inline

In [4]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt # ! is a special syntax which lets us run shell commands directly in the collab notebook, wget is a command line utility which downloads files form the internet.


--2026-09-24 16:41:45--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.03s   

2026-09-24 16:41:45 (7.78 MB/s) - ‘names.txt’ saved [228145/228145]



In [5]:
words = open('names.txt', 'r').read().splitlines() # it uses an inbuilt method called open which takes the name of the file and a parameter called 'r' to open in read mode and gives a method to read which returns the contents of the file in a string then we split each string on new line as each element of an array.


In [6]:
# build the vocabulary of character and mappings to/from integers
chars = sorted(list(set(''.join(words)))) # gets the set of sorted list of words the join takes each words in the array and creates a string then the set dedups the repeating chars essentially at max it will give a to z chars then list makes it an array and we sort that array to get a to z literally
stoi = {s:i+1 for i,s in enumerate(chars)} #
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [7]:
# build the dataset
block_size = 8 # context length: how many characters do we take to predict the next one

# function to build dataset - it compiles the data into train, dev and test split in a 80, 10, 10 split
def build_dataset(words): # function definition with positional arguments (accepts list of words)
  X, Y = [], [] # X and Y lists here X will store the training input and Y will store its respective label to find loss and or verify correctness

  for w in words: # loop over words list iterating each word e.g w = "emma"
    context = [0] * block_size # context length input sequence [0] * 3 = [0, 0, 0]
    for ch in w + '.': # loop over each character in the current word w e.g ch = 'e'
      ix = stoi[ch] # integer mapping of this character from our vocabulary stoi e.g ix = 5
      X.append(context) # append the context for this char in X list
      Y.append(ix) # append the label for this char, together it looks like [0,0,0] -> 5 meaning for '...' output 'e' as in e should follow start of a word
      context = context[1:] + [ix] # remove first index from context and take current char as last index to make the window move to right for next char prediction input

  X = torch.tensor(X) # convert X list into a tensor - useful for tensor manipulations like dot product and normalizations
  Y = torch.tensor(Y) # convert Y list into a tensor - it will not change the values inside X and Y as tensor is just a way of representation.
  print(X.shape, Y.shape) # print shape of X tensor and Y tensor
  return X, Y # return the X and Y tensors

import random # random module in python used to get functions related to random outcomes
random.seed(42) # manual seed of 42 for predictability in the dataset over multiple runs
random.shuffle(words) # shuffle the words randomly.
n1 = int(0.8*len(words)) # to get 0 to 80% for train
n2 = int(0.9*len(words)) # to get 80 to 90% for dev and then 90 to 100% for test

Xtr, Ytr = build_dataset(words[:n1]) # 80%
Xdev, Ydev = build_dataset(words[n1:n2]) # 10%
Xte, Yte = build_dataset(words[n2:]) # 10%

torch.Size([182625, 8]) torch.Size([182625])
torch.Size([22655, 8]) torch.Size([22655])
torch.Size([22866, 8]) torch.Size([22866])


In [8]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

........ --> y
.......y --> u
......yu --> h
.....yuh --> e
....yuhe --> n
...yuhen --> g
..yuheng --> .
........ --> d
.......d --> i
......di --> o
.....dio --> n
....dion --> d
...diond --> r
..diondr --> e
.diondre --> .
........ --> x
.......x --> a
......xa --> v
.....xav --> i
....xavi --> e


In [9]:
# Let's train a deeper network
class Linear: # defines a class named Linear (a layer is called a Linear here before activation function is applied to it)

   # __init__ function which runs when Linear() is called, it takes in some parameters as follows
   # fan_in: the number of weights in a single neuron to watch input, it is equal to the input size e.g if 30 then 30 rows meaning 30 weights for the first column denotes a neuron here
   # fan_out: the number of neurons in the entire layer e.g 200 so the layer will have (30, 200) meaning 200 neurons each neuron having 30 weights.
   # bias: a boolean flag for bias to be initialised or not .
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5 # all the neurons divided by the square root of number of weights to take the mean - maybe ?? (some batch norm thing going on here ??)
    self.bias = torch.zeros(fan_out) if bias else None # bias if needed else none, for bias we take 0 for all the neurons say 200

  # __call__ function is a function which runs when the initialized class instance object runs example layer() - this call function is used to do a dot product with weights when input is passed of matching shape
  # Parameters:
  # x: the input (could be the input to the hidden layer or to the output layer)
  def __call__(self, x):
    self.out = x @ self.weight # dot product inout with weights of this layer (30, 200) all 200 neurons multiplies with input here input will be of (R, 30) there will be R examples of each 30 features or 30 vectors to be watched by the 30 weights in 200 neurons
    if self.bias is not None: # if bias is not none then add them
      self.out += self.bias # add the bias to the output of dot product
    return self.out # return the pre activation computation of this layer wrt to the given input

  # Returns all the parameters of this layer
  def parameters(self): # no arguments
    return [self.weight] + ([] if self.bias is None else [self.bias]) # returns all the neurons with its weights and bias if bias is there in a list

class BatchNorm1d: # defines a class to hold batch norm variables as in the running mean, variance, bias and gain etc - ?? (batch norm is not so clear to me)

  # Params:
  # dim: the dimensions - ??
  # eps: - ??
  # momentum: - ??
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps # ??
    self.momentum = momentum # ??
    self.training = True # for training or not (as in if backprop will be called or not) - ??
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim) # knobs to adjust variance and mean ??
    self.beta = torch.zeros(dim) # knobs to adjust variance and mean ??
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim) # needed at inference - not sure why ??
    self.running_var = torch.ones(dim) # needed at inference - not sure why ??

  # params:
  # x: the input passed to the batchnorm ??
  def __call__(self, x):
    # calculate the forward pass
    if self.training: # if backprop will be called as in trained
      xmean = x.mean(0, keepdim=True) # batch mean - to normalize the large values to be at centre or something - not sure, confusion here ??
      xvar = x.var(0, keepdim=True, unbiased=True) # batch variance - ??
    else: # for not training as in where backprop won't be called
      xmean = self.running_mean # collect running mean which is being updated ??
      xvar = self.running_var # collect running variance which is being updated ??
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance - not sure what is happening here ??
    self.out = self.gamma * xhat + self.beta # ??
    # update the buffers
    if self.training: # we need to update the running mean and var
      with torch.no_grad(): # in these mathematical expressions we do not want to call backprop so no grad is required here
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean # ??
        self.riunning_var = (1 - self.momentum) * self.running_var + self.momentum * xvar # ??
    return self.out # return the final output

  # returns all the parameters of the batchnorm - it is being trained too.
  def parameters(self): # no args
    return [self.gamma, self.beta] # gamma and beta are the two knobs

# defines the activation function class
class Tanh:
  # function to apply activation on input
  def __call__(self, x): # takes input x
    self.out = torch.tanh(x) # returns the squashed output for a given input, performs tanh
    return self.out # returns the output

  # function to return all the parameters of this function - since everything is trained in a nn mostly (nn is nothing but a giant mathematical expression)
  def parameters(self): # no args
    return [] # no params but sure is derivated in backprop

class Embedding: # defines a class for embeddings example C is an embedding of randn so we utilize this class as in C is now Embedding

  def __init__(self, num_embeddings, embedding_dim): # args: num_embeddings is number of examples / rows, embedding_dim is the dimensions along each example
    self.weight = torch.randn((num_embeddings, embedding_dim)) # create a random tensor of given rows and cols - here since it resembles character embedding we will have 27 rows from a to z and '.' and each once will be a vector of 10 dimension so (27,10)

  def __call__(self, IX): # args: IX - it is the random int to be plucked out from Embedding / C
    self.out = self.weight[IX] # return the random examples out example say the IX is Xb = Xtr[ix] which is [32,3] here 32 is batch size and 3 is the 3 integers for 3 different characters basically the contenxt length then it plucks out 32 rows of 3 10-dim vectors representing each char so (32, 3, 10)
    return self.out # returns the embedding for (32,3,10)

  def parameters(self): # no args - returns parameters to be trained
    return [self.weight] # returns the parameters

class FlattenConsecutive: # defines the class to concatenate the embedding to be passed into nn as input

  def __init__(self, n): # args: n: number of consecutive characters to be concatenated example pair of 2
    self.n = n # set the n to be used in view operation

  def __call__(self, x): # args: x is the embedding to be concatenated example on shape (32,3, 10) -> (32, 30) example
    B, T, C = x.shape # get the shape of the input e.g 4, 8, 10 => 4 examples of 8 context length as in char and each of 10 dimension now we need (4, 4, 20) 4 examples in 4 pairs each pair concatenated to make 20 vectors
    x = x.view(B, T//self.n, C*self.n) # gives (examples, pairs = (context) / n = 8/2 = 4, 10 * 2) => (4, 4, 20) for instance.
    if x.shape[1] == 1: # if the shape of second dim of input became 1 for some reason then we just squeeze that dimension
      x = x.squeeze(1) # squeeze that dim as in concatenate
    self.out = x # set the input to out
    return self.out # return the output (4, 4, 20) concatenated

  def parameters(self): # returns the parameters
    return [] # nothing to be trained

class Sequential: # defines a container class which takes in all the layers and performs desired computation

  def __init__(self, layers): # args: layers: all the layers in nn
    self.layers = layers # set the layers

  def __call__(self, x): # args: x: the input to be passed to the layer
    for layer in self.layers: # loop over in all the layer
      x = layer(x) # get the output of this layer for this input
    self.out = x # set it as out
    return self.out # return out - it can be preact, activation result, batchnorm, etc

  def parameters(self): # no args
    return [p for layer in self.layers for p in layer.parameters()] # return all the parameters of all the layers


In [19]:
n_embd = 10 # the dimensionality of the character embedding vector - for 3 char each char will be a an integer mapping which then will be a 10 dimension vector each so 3 * 10  = 30 vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP, layer shape will be (30, 100)
g = torch.Generator().manual_seed(2147483647) # for reproducibility

model = Sequential([
    Embedding(vocab_size, n_embd),
    FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size)
])

# with torch.no_grad():
#   # last layer: make less confident
#   layers[-1].weight *= 0.1

parameters = model.parameters()
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

170897


In [20]:
ix = torch.randint(0, Xtr.shape[0], (4,), generator=g) # 0 to shape[0] rand int and 4 examples
Xb, Yb = Xtr[ix], Ytr[ix]
logits = model(Xb)
print(Xb.shape)
Xb

torch.Size([4, 8])


tensor([[ 0,  0,  0,  0,  0,  0,  1, 16],
        [ 0,  0, 22,  9, 19,  8,  1,  1],
        [ 0,  0,  0,  0,  0,  0,  2,  1],
        [ 0,  0,  0,  0,  0,  0,  0,  0]])

In [21]:
for layer in model.layers:
  print(layer.__class__.__name__, ':', tuple(layer.out.shape))

Embedding : (4, 8, 10)
FlattenConsecutive : (4, 4, 20)
Linear : (4, 4, 200)
BatchNorm1d : (4, 4, 200)
Tanh : (4, 4, 200)
FlattenConsecutive : (4, 2, 400)
Linear : (4, 2, 200)
BatchNorm1d : (4, 2, 200)
Tanh : (4, 2, 200)
FlattenConsecutive : (4, 400)
Linear : (4, 200)
BatchNorm1d : (4, 200)
Tanh : (4, 200)
Linear : (4, 27)


In [11]:
# Same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]

  # forward pass
  logits = model(Xb)
  loss = F.cross_entropy(logits, Yb) # loss function

  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

  break # AFTER DEBUGGING would take out obviously to run full optimization


      0/ 200000: 3.6702


In [12]:
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))

RuntimeError: shape '[-1, 1000]' is invalid for input of size 1

In [13]:
for layer in model.layers:
  layer.training = False


In [14]:
# evaluate the loss
@torch.no_grad() # this decorator disables gradient tracking inside pytorch
def split_loss(split): # args: split key to return the appropriate split e.g 'train'
  x,y = { # object of key to split of data
      'train': (Xtr, Ytr), # train split
      'val': (Xdev, Ydev), # val split
      'test': (Xte, Yte) # test split
  }[split] # index the object on the key passed
  logits = model(x) # get the logits from the input passing in the model
  loss = F.cross_entropy(logits, y) # get the loss from the logits and output label
  print(split, loss.item()) # print the loss wrt split key

split_loss('train') # split loss on train - forwards train input and gets loss
split_loss('val') # split loss on val - forwards val input and gets loss

train 3.4208056926727295
val 3.4140121936798096


In [15]:
# sample from the model

for _ in range(20): # loop over 2o times to sample 20 names

  out = [] # collect the characters to be joined as a name when '.' is encountered
  context = [0] * block_size # initialize with all '...'
  while True: # run indefinitely as the break is when '.' appears
    # forward pass
    logits = model(torch.tensor([context])) # input the context to start off the prediction - it will perform forward and gives the logits
    probs = F.softmax(logits, dim=1) # get the normalized probabilities using softmax
    # sample from the distribution
    ix = torch.multinomial(probs, num_samples=1).item() # get the multinomail distributin from the probs (most prob gets higher chance to be picked like 3 red 2 blue 1 green) - chances to be picked: red > blue > green
    # shift the context window and track the samples
    context = context[1:] + [ix] # remove first char take next - sliding window
    out.append(ix) # append the character predicted
    # if we sample the special '.' token, break
    if ix == 0: # 0 is mapped to '.' and '.' do not follow anyone as the end so it is the ending of a word or a name
      break # break out of the infinite while loop

  print(''.join(itos[i] for i in out))



coadmtwthmbeyzgekqtvnmbsacljrgexmopiylmodyyyopvwjbeavmwkvhbpvhjjssmrs.
yuutdbnjgjrsyfaqtgtnyjgqcngubgrxaxqm.
vyxixopsa.
yyivpvtzdxbmpdfqsfhygwglsxcckzyjligjcqvruwzlfhabvhlpmxjhhsq.
hrinwdtbsxkhtbsepoobp.
bdvoidlhjzdvjdkhfpjuptgtqmxfdvjqzscepdvfxihmxwdpzapwnqkhmodtekdwxyulykgdbmheuikxehyptjlxdjmfouchrpl.
h.
phvm.
zphu.
fjslfyfleommdhdaxbqxqafvevtmxfgdgwnzokkjfhfishyzbbbiwzvzdaadscsfhpwymzsyhuxyxppdbndwgafhqctg.
hivzufbgulowthdxzrfhswihmxrbhahjlnjuchgcae.
xba.
d.
.
tywhezjbupchz.
dibgfpbj.
thacav.
vhihzytkortcfqogtzvtycxmbowkvifppdnkdbvohajqbzrwyeljpxikyhohnjgfbgakhngmgxfjggbg.
orihfeibfhzeju.
lacclxhdjmgnasopghfzjqgobjwmoilowqkdrqyfzy.
